## Set Up

In [ ]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import GenerationConfig
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel
import warnings
import torch
import os
import warnings

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
warnings.filterwarnings("ignore")
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/MyDrive/Colab Notebooks/efsa_sentiment_classification.csv'
df = pd.read_csv(file_path)

# Extract test
df = df[939:].reset_index(drop=True)
length = len(df)

### Load Model

In [ ]:
SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

FINE_TO_COARSE = {}
FINE_EVENT_LIST = []

for coarse_key, fine_list in FINANCIAL_EVENT_LIST.items():
    for fine_event in fine_list:
        FINE_TO_COARSE[fine_event] = coarse_key
        FINE_EVENT_LIST.append(fine_event)

coarse_event_definitions = """
- Financial Affairs: Earnings announcements, forecasts, or other financial updates.
- Shareholder Affairs: Actions by shareholders such as pledges, stock adjustments.
- Stock Affairs: Stock buybacks, dividends, price movement announcements.
- Business Operations: New products, partnerships, quality control, or business activities.
- Compliance and Credit: Legal issues, investigations, rating changes.
- Management Affairs: Executive hires, promotions, or organizational changes.
- Financing and Investment: Mergers, acquisitions, funding rounds, IPOs.
"""

In [ ]:
INDUSTRY_LIST = ["Energy Equipment & Services", "Oil, Gas & Consumable Fuels", "Chemicals", "Construction Materials", "Containers & Packaging", "Metals & Mining", "Paper & Forest Products", "Aerospace & Defense", "Building Products", "Construction & Engineering", "Electrical Equipment", "Industrial Conglomerates", "Machinery", "Trading Companies & Distributors", "Commercial Services & Supplies", "Professional Services", "Air Freight & Logistics", "Passenger Airlines", "Marine Transportation", "Ground Transportation", "Transportation Infrastructure", "Automobile Components", "Automobiles", "Household Durables", "Leisure Products", "Textiles, Apparel & Luxury Goods", "Hotels, Restaurants & Leisure", "Diversified Consumer Services", "Distributors", "Broadline Retail", "Specialty Retail", "Consumer Staples Distribution & Retail", "Beverages", "Food Products", "Tobacco", "Household Products", "Personal Care Products", "Health Care Equipment & Supplies", "Health Care Providers & Services", "Health Care Technology", "Biotechnology", "Pharmaceuticals", "Life Sciences Tools & Services", "Banks", "Financial Services", "Consumer Finance", "Capital Markets", "Mortgage Real Estate Investment Trusts (REITs)", "Insurance", "IT Services", "Software", "Communications Equipment", "Technology Hardware, Storage & Peripherals", "Electronic Equipment, Instruments & Components", "Semiconductors & Semiconductor Equipment", "Diversified Telecommunication Services", "Wireless Telecommunication Services", "Media", "Entertainment", "Interactive Media & Services", "Electric Utilities", "Gas Utilities", "Multi-Utilities", "Water Utilities", "Independent Power and Renewable Electricity Producers", "Diversified REITs", "Industrial REITs", "Hotel & Resort REITs", "Office REITs", "Health Care REITs", "Residential REITs", "Retail REITs", "Specialized REITs", "Real Estate Management & Development"]

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 109.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:
#Quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# Loading Mistral with 4-bit quantization
from transformers import GenerationConfig

MISTRAL_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading mistral model and tokenizer")
mistral_tokenizer = AutoTokenizer.from_pretrained(MISTRAL_MODEL_NAME, trust_remote_code=True)
mistral_model = AutoModelForCausalLM.from_pretrained(
    MISTRAL_MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quantization_config,  # add Quantization
    torch_dtype=torch.bfloat16
)

print("Mistral Model loaded successfully!")

Loading mistral model and tokenizer


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Mistral Model loaded successfully!


In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel

print("Loading model and tokenizer...")
lora_base_model = AutoModelForCausalLM.from_pretrained(MISTRAL_MODEL_NAME, device_map='auto', torch_dtype=torch.bfloat16, quantization_config=quantization_config)

lora_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer", use_fast=True)
lora_model = PeftModel.from_pretrained(lora_base_model, "/content/drive/MyDrive/Colab Notebooks/mistral_lora_model")

print('Model loaded successfully!')

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Prompt Engineering

### Zero Shot

In [ ]:
# ==================== ZERO SHOT PROMPT TEMPLATES ====================

ZERO_SHOT_PROMPTS = {
    "extract_companies": """Financial text: {text}

Extract ALL company or ticker names mentioned in this text.
If multiple companies are mentioned, separate them with commas.
Only answer with unique company names, without additional text or punctuation.""",

#----------------------

    "get_industry": """Text: {text}
Company: {company_name}

Classify {company_name} into the most appropriate GICS sector.
Available GICS sectors: {industry_list}
Respond only with an industry name from the list, without additional text or punctuation.""",

#-----------------------

    "classify_coarse_event": """Text: "{text}"
Company: {company_name}
Industry: {industry}

Choose the most appropriate coarse-grained event type from this list based on stated facts in the text: {coarse_events}

Answer only with one coarse-grained event type, without additional text or punctuation.""",
#----------------------

    "classify_fine_event": """Coarse-Grained Event Classification: {coarse_event}.
Text: "{text}"
Company: {company_name}
Industry: {industry}

Determine the specific type of corporate event that occurred at {company_name}.
Options: {fine_events}
If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.
Respond only with an event type from the provided list, without additional text or punctuation.""",

#----------------------
    "analyze_sentiment": """Analyze market sentiment for this text.

Company: {company_name} ({industry})
Event: {fine_event}
News: {text}

Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."""
}


### Few Shot

In [ ]:
# ==================== FEW SHOT PROMPT TEMPLATES ====================

FEW_SHOT_PROMPTS = {
    "extract_companies": """Financial text: {text}

Extract ALL company or ticker names mentioned in this text.
If multiple companies are mentioned, separate them with commas.
Only answer with unique company names in full, without additional text or punctuation.

Here are some examples:

Example 1:
Financial text: Royal Mail chairman Donald Brydon set to step down
Answer: Royal Mail

Example 2:
Financial text: AstraZeneca Acquires ZS Pharma in $2.7 Billion Deal
Answer: ZS Pharma

Example 3:
Financial text: Barclays sells benchmark indices unit to Bloomberg
Answer: Bloomberg""",

# ------------

    "get_industry": """Text: {text}
Company: {company_name}

Classify {company_name} into the most appropriate GICS sector.
Available GICS sectors: {industry_list}
Respond only with an industry name from the list, without additional text or punctuation.

Here are some examples:

Example 1:
Text: AstraZeneca's Lung Cancer Drug Tagrisso Gets FDA Approval
Company: AstraZeneca
Answer: Pharmaceuticals

Example 2:
Text: Bunzl Lifts Dividend Again As Acquisitions Continue To Boost
Company: Bunzl
Answer: Consumer Staples Distribution & Retail

Example 3:
Text: EU drops Shell, BP, Statoil from ethanol benchmark investigation
Company: Statoil
Answer: Oil, Gas & Consumable Fuels""",

# ------------

    "classify_coarse_event": """Text: "{text}"
Company: {company_name}
Industry: {industry}

Choose the most appropriate coarse-grained event type from this list based on stated facts in the text: {coarse_events}

Here are some examples:

Example 1:
Financial text: Royal Mail chairman Donald Brydon set to step down
Answer: Royal Mail

Example 2:
Financial text: AstraZeneca Acquires ZS Pharma in $2.7 Billion Deal
Answer: ZS Pharma

Example 3:
Financial text: Microsoft and Google announced a strategic partnership
Answer: Microsoft, Google

Answer only with one coarse-grained event type, without additional text or punctuation.""",
# ------------

    "classify_fine_event": """Coarse-Grained Event Classification: {coarse_event}.
Text: "{text}"
Company: {company_name}
Industry: {industry}

Determine the specific type of corporate event that occurred at {company_name}.
Options: {fine_events}

Here are some examples:

Example 1:
Text: "Apple reported quarterly earnings that beat analyst estimates"
Company: Apple
Industry: Technology
Coarse Event: Financial Affairs
Fine Events: ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"]
Answer: Profit Announcement

Example 2:
Text: "Tesla launched a stock buyback program worth $5 billion"
Company: Tesla
Industry: Consumer Discretionary
Coarse Event: Stock Affairs
Fine Events: ["Stock Price Movement", "Stock Buyback", "Stock Dividend", "Other Stock Affairs"]
Answer: Stock Buyback

Example 3:
Text: "Lloyds to cut 945 jobs as part of three-year restructuring strategy"
Company: Lloyds
Industry: Banks
Coarse Event: Management Affairs
Fine Events: ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"]
Answer: Employee Dynamics

If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.
Respond only with an event type from the provided list, without additional text or punctuation.""",


# ------------

    "analyze_sentiment": """Analyze market sentiment for this text.

Company: {company_name} ({industry})
Event: {fine_event}
News: {text}

Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation.

Here are some examples:

Example 1:
Company: Morrisons (Consumer Staples Distribution & Retail)
Event: Stock Price Movement
News: Morrisons and Debenhams surprise City with Christmas trading
Answer: POS

Example 2:
Company: Baxalta (Pharmaceuticals)
Event: Stock Price Movement
News: Shire share price under pressure after $32bn Baxalta takeover rejected
Answer: NEG

Example 3:
Company: Shell (Oil, Gas & Consumable Fuels)
Event: Initiating Cooperation
News: Shell and BG Shareholders to Vote on Deal at End of January
Answer: NEU"""
}


## Build CoT

In [ ]:
logger = logging.getLogger(__name__)

def get_current_model_and_tokenizer(method: str):
    """
    Select appropriate model and tokenizer based on the specified method.

    Args:
        method: One of "zero_shot", "few_shot", or "lora"

    Returns:
        tuple: (model, tokenizer, model_name) for the specified method
    """
    if method in ["zero_shot", "few_shot"]:
        return mistral_model, mistral_tokenizer, "mistral"
    elif method == "lora":
        return lora_model, lora_tokenizer, "mistral"
    else:
        raise ValueError("Method must be 'zero_shot', 'few_shot', or 'lora'")


###  Build CoT Step Functions

In [ ]:
def query_model(model, model_name: str, tokenizer, prompt: str, temperature: int, max_new_tokens=16) -> str:
    """
    Generate model response for given prompt using appropriate format.

    Args:
        model: The language model to use
        model_name: Name/type of the model (mistral or llama)
        tokenizer: Corresponding tokenizer
        prompt: Input prompt for the model
        max_new_tokens: Maximum number of tokens to generate

    Returns:
        str: Generated response from the model
    """
    # Format prompt according to model type
    instruction = f"[INST] {prompt} [/INST]"

    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():  # Disable gradient computation for inference
            output_ids = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # Use greedy decoding for faster inference
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True  # Enable KV cache for speed
            )

        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Extract only the generated part (remove input prompt)
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""


def extract_companies(text: str, method="zero_shot") -> List[str]:
    """
    Extract company names from financial text using specified method.

    Args:
        text: Input financial text
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        List[str]: List of extracted company names
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["extract_companies"].format(text=text)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["extract_companies"].format(text=text)
    elif method == "lora":
        prompt = ZERO_SHOT_PROMPTS["extract_companies"].format(text=text)

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=16, temperature=0.2)
        # Parse comma-separated company names
        companies = [company.strip() for company in response.split(',') if company.strip()]
        return companies if companies else ["Unknown Company"]
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return ["Unknown Company"]

def get_industry(text: str, company_name: str, method="zero_shot") -> str:
    """
    Classify company industry using specified method.

    Args:
        text: Context text containing company information
        company_name: Name of the company to classify
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted industry classification
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["get_industry"].format(
            text=text, company_name=company_name, industry_list=INDUSTRY_LIST)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["get_industry"].format(
            text=text, company_name=company_name, industry_list=INDUSTRY_LIST)
    elif method == "lora":
        prompt = ZERO_SHOT_PROMPTS["get_industry"].format(
            text=text, company_name=company_name, industry_list=INDUSTRY_LIST)

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=8, temperature=0.1)
        # Clean response from quotes and extra characters
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"

def classify_coarse_grained_event(text: str, industry: str, company_name: str, method="zero_shot") -> str:
    """
    Classify coarse-grained financial event type.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted coarse-grained event type
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    coarse_events = list(FINANCIAL_EVENT_LIST.keys())

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["classify_coarse_event"].format(
            text=text, company_name=company_name, industry=industry, coarse_events=coarse_events)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["classify_coarse_event"].format(
            text=text, company_name=company_name, industry=industry, coarse_events=coarse_events)
    elif method == "lora":
        prompt = ZERO_SHOT_PROMPTS["classify_coarse_event"].format(
            text=text, company_name=company_name, industry=industry, coarse_events=coarse_events)

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=8, temperature=0.1)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Event"
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event"

def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str, method="zero_shot") -> str:
    """
    Classify fine-grained financial event type.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        coarse_event: Previously classified coarse event
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted fine-grained event type
    """
    # Check if coarse event is valid
    if coarse_event not in FINANCIAL_EVENT_LIST:
        return "Unknown Fine Event"

    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    fine_events = FINANCIAL_EVENT_LIST[coarse_event]

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["classify_fine_event"].format(
            text=text, company_name=company_name, industry=industry,
            coarse_event=coarse_event, fine_events=fine_events)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["classify_fine_event"].format(
            text=text, company_name=company_name, industry=industry,
            coarse_event=coarse_event, fine_events=fine_events)
    elif method == "lora":
        prompt = ZERO_SHOT_PROMPTS["classify_fine_event"].format(
            text=text, company_name=company_name, industry=industry,
            coarse_event=coarse_event, fine_events=fine_events)

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=8, temperature=0.15)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Fine Event"
    except Exception as e:
        logger.error(f"Error classifying fine event: {e}")
        return "Unknown Fine Event"


def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str, method="zero_shot") -> str:
    """
    Analyze sentiment polarity of financial news.

    Args:
        text: Financial news text
        industry: Company's industry
        company_name: Name of the company
        coarse_event: Coarse-grained event type
        fine_event: Fine-grained event type
        method: Analysis method ("zero_shot", "few_shot", or "lora")

    Returns:
        str: Predicted sentiment (POS, NEG, or NEU)
    """
    # Get appropriate model and tokenizer for the method
    model, tokenizer, model_name = get_current_model_and_tokenizer(method)

    # Choose prompt based on method
    if method == "zero_shot":
        prompt = ZERO_SHOT_PROMPTS["analyze_sentiment"].format(
            text=text, company_name=company_name, industry=industry, fine_event=fine_event)
    elif method == "few_shot":
        prompt = FEW_SHOT_PROMPTS["analyze_sentiment"].format(
            text=text, company_name=company_name, industry=industry, fine_event=fine_event)
    elif method == "lora":
        prompt = ZERO_SHOT_PROMPTS["analyze_sentiment"].format(
            text=text, company_name=company_name, industry=industry, fine_event=fine_event)

    try:
        response = query_model(model, model_name, tokenizer, prompt, max_new_tokens=4, temperature=0.05)
        return response.strip() or "Unknown"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "Unknown"


### Build CoT Analyze Fucntion

In [ ]:
def analyze_financial_text(text: str, method="zero_shot") -> List[Tuple]:
    try:
        # Step 1: Extract all company names from the text
        company_names = extract_companies(text, method)
        results = []

        # Step 2: Analyze each company mentioned in the text
        for company_name in company_names:
            try:
                # Get industry classification
                industry = get_industry(text, company_name, method)

                # Classify coarse-grained event
                coarse_event = classify_coarse_grained_event(text, industry, company_name, method)

                # Classify fine-grained event based on coarse event
                fine_event = classify_fine_grained_event(text, industry, company_name, coarse_event, method)

                # Analyze sentiment polarity
                sentiment = analyze_sentiment(text, industry, company_name, coarse_event, fine_event, method)

                # Store complete analysis result
                results.append((text, company_name, industry, coarse_event, fine_event, sentiment))

            except Exception as e:
                logger.error(f"Error analyzing text for company {company_name}: {e}")
                # Add error entry to maintain data structure
                results.append((text, company_name, "Error", "Error", "Error", "Error"))

        return results

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return [(text, "Error", "Error", "Error", "Error", "Error")]

### Build Function for Result Data Frame

In [ ]:
def run_three_method_experiment(df, num_samples=20):
    """
    Run comparative experiment using all three methods.

    Args:
        df: DataFrame containing financial texts
        num_samples: Number of samples to process

    Returns:
        tuple: (zero_df, few_df, lora_df) containing results from each method
    """
    print(f"Running three-method experiment on {num_samples} samples...")

    # Initialize result containers
    results_zero = []
    results_few = []
    results_lora = []

    # Limit dataset size for testing
    test_df = df[:num_samples]

    # Process each text with all three methods
    for idx, row in test_df.iterrows():
        text = row['Sentence']

        if pd.isna(text) or text.strip() == '':
            logger.warning(f"Empty text at index {idx}")
            # Add empty results for consistency
            results_zero.append([("", "", "", "", "", "")])
            results_few.append([("", "", "", "", "", "")])
            results_lora.append([("", "", "", "", "", "")])
            continue

        print(f"Processing {idx+1}/{len(test_df)}: {text[:50]}...")

        try:
            # Zero-shot analysis
            result_zero = analyze_financial_text(text, method="zero_shot")
            results_zero.append(result_zero)

            # Few-shot analysis
            result_few = analyze_financial_text(text, method="few_shot")
            results_few.append(result_few)

            # LoRA analysis
            result_lora = analyze_financial_text(text, method='lora')
            results_lora.append(result_lora)

        except Exception as e:
            logger.error(f"Error processing sample {idx}: {e}")
            # Add error results
            error_result = [(text, "Error", "Error", "Error", "Error", "Error")]
            results_zero.append(error_result)
            results_few.append(error_result)
            results_lora.append(error_result)

        # Progress update every 5 samples
        if (idx + 1) % 5 == 0:
            print(f" Completed {idx + 1}/{len(test_df)} texts")
            # Clear GPU cache periodically
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Flatten results and create DataFrames
    columns = ['Sentence', 'Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment']

    flat_zero = [item for sublist in results_zero for item in sublist]
    flat_few = [item for sublist in results_few for item in sublist]
    flat_lora = [item for sublist in results_lora for item in sublist]

    zero_df = pd.DataFrame(flat_zero, columns=columns)
    few_df = pd.DataFrame(flat_few, columns=columns)
    lora_df = pd.DataFrame(flat_lora, columns=columns)

    print("\\nThree-method experiment completed!")
    print(f"Zero-shot results: {len(zero_df)} entries")
    print(f"Few-shot results: {len(few_df)} entries")
    print(f"LoRA results: {len(lora_df)} entries")

    return zero_df, few_df, lora_df

In [ ]:
zero_df, few_df, lora_df = run_three_method_experiment(df, num_samples=length)

Running three-method experiment on 234 samples...
Processing 1/234: $TSLA TO RECALL 2,700 MODEL X SUVS FOR SEAT-BELT F...
Processing 2/234: 5 Best Analyst Rated Stocks in Last 72hrs: $ORCL $...
Processing 3/234: Bought some more $CELG as it is ready for a bounce...
Processing 4/234: Analysts impressed with progress at Tesla's flagsh...
Processing 5/234: $CELG back near 104, where it opened and rallied h...
 Completed 5/234 texts
Processing 6/234: $AMZN new HOD with conviction keeping $570 on watc...
Processing 7/234: $FB  rejecting HIGHS shortable...at 109...
Processing 8/234: $CSX is up today to report.  Wall Street is expect...
Processing 9/234: $FB turns back down in early trading.... https://t...
Processing 10/234: Followed the levels I shared with you on $NFLX $GO...
 Completed 10/234 texts
Processing 11/234: $FAST $GWW - daily sales slowing again, pretty tim...
Processing 12/234: Tesla Motors recalls 2,700 Model X SUVs $TSLA http...
Processing 13/234: SiriusXM added 465,000 net n

## Evaluation

### F1 Function for Single Label

In [ ]:
from collections import defaultdict, Counter
from sklearn.metrics import f1_score, accuracy_score
def calculate_eval_metrics_by_label(df, results_df):
    """
    Calculate weighted F1 and accuracy for each category.
    """
    preds_by_sentence = results_df.groupby('Sentence')
    categories = ['Company', 'Sentiment', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event']
    gt_labels = defaultdict(list)
    pred_labels = defaultdict(list)
    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()
        if sentence not in preds_by_sentence.groups:
            for cat in categories:
                gt_labels[cat].append(str(row[cat]))
                pred_labels[cat].append('NO_PREDICTION')
            continue
        preds = preds_by_sentence.get_group(sentence)
        company_match = preds[preds['Company'].str.lower().str.contains(gt_company, na=False)]
        if company_match.empty:
            for cat in categories:
                gt_labels[cat].append(str(row[cat]))
                pred_labels[cat].append('NO_COMPANY_MATCH')
            continue
        pred_row = company_match.iloc[0]
        for cat in categories:
            gt_val = str(row[cat])
            pred_val = str(pred_row[cat])
            gt_labels[cat].append(gt_val)
            pred_labels[cat].append(pred_val)
    # Compute all metrics
    results = {}
    for cat in categories:
        y_true = gt_labels[cat]
        y_pred = pred_labels[cat]
        labels = list(set(y_true + y_pred))
        # Calculate accuracy
        accuracy = accuracy_score(y_true, y_pred)
        # Calculate F1 scores
        weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=labels, zero_division=0)
        results[cat] = {
            'accuracy': round(accuracy, 4),
            'f1_weighted': round(weighted_f1, 4)
        }
    return results

### F1 Function for Tuple Labels

In [ ]:
from sklearn.metrics import f1_score
from collections import defaultdict
def calculate_eval_metrics_by_variant(df, results_df):
    """
    Calculate weighted F1 and accuracy for EFSA variants.
    """
    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse-Grained Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine-Grained Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment']
    }
    preds_by_sentence = results_df.groupby('Sentence')
    lab_true = defaultdict(list)
    lab_pred = defaultdict(list)
    for _, row in df.iterrows():
        sentence = row['Sentence']
        gt_company = str(row['Company']).lower()
        if sentence not in preds_by_sentence.groups:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_PREDICTION'] * len(cols)))
            continue
        preds = preds_by_sentence.get_group(sentence)
        company_match = preds[preds['Company'].str.lower().apply(
            lambda x: gt_company in x or x in gt_company)]
        if company_match.empty:
            for variant, cols in variants.items():
                lab_true[variant].append("|".join(str(row[col]) for col in cols))
                lab_pred[variant].append("|".join(['NO_COMPANY_MATCH'] * len(cols)))
            continue
        pred_row = company_match.iloc[0]
        for variant, cols in variants.items():
            gt_string = "|".join(str(row[col]) for col in cols)
            pred_string = "|".join(str(pred_row[col]) for col in cols)
            lab_true[variant].append(gt_string)
            lab_pred[variant].append(pred_string)
    results = {}
    for variant in variants:
        true = lab_true[variant]
        pred = lab_pred[variant]
        labels = list(set(true + pred))
        # Calculate accuracy
        accuracy = accuracy_score(true, pred)
        # Calculate F1 scores
        weighted_f1 = f1_score(true, pred, average='weighted', labels=labels, zero_division=0)
        results[variant] = {
            'accuracy': round(accuracy, 4),
            'f1_weighted': round(weighted_f1, 4)
        }
    return results

### Comparison of Evaluation

In [ ]:
def compare_three_methods(df, zero_df, few_df, lora_df):
    """Compare Zero-shot, Few-shot, and LoRA methods performance."""

    # Calculate metrics for all methods
    zero_by_label = calculate_eval_metrics_by_label(df, zero_df)
    few_by_label = calculate_eval_metrics_by_label(df, few_df)
    lora_by_label = calculate_eval_metrics_by_label(df, lora_df)

    zero_by_variant = calculate_eval_metrics_by_variant(df, zero_df)
    few_by_variant = calculate_eval_metrics_by_variant(df, few_df)
    lora_by_variant = calculate_eval_metrics_by_variant(df, lora_df)

    # Function to determine best method
    def get_best_method(zero_score, few_score, lora_score):
        best_score = max(zero_score, few_score, lora_score)
        if best_score == zero_score:
            return "Zero-shot"
        elif best_score == few_score:
            return "Few-shot"
        else:
            return "LoRA"

    print("=" * 100)
    print("THREE-METHOD COMPARISON - ACCURACY")
    print("=" * 100)
    print(f"{'Task':<30} {'Zero-shot':<12} {'Few-shot':<12} {'LoRA':<12} {'Best':<12}")
    print("-" * 100)

    # Label-based tasks - Accuracy
    for task in ['Company', 'Sentiment', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event']:
        zero_score = zero_by_label[task]['accuracy']
        few_score = few_by_label[task]['accuracy']
        lora_score = lora_by_label[task]['accuracy']

        best = get_best_method(zero_score, few_score, lora_score)
        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

    print("-" * 100)

    # Variant-based tasks - Accuracy
    for task in ['EFSA', 'Coarse-grained EFSA', 'Fine-grained EFSA', 'Entity-Level FSA']:
        zero_score = zero_by_variant[task]['accuracy']
        few_score = few_by_variant[task]['accuracy']
        lora_score = lora_by_variant[task]['accuracy']

        best = get_best_method(zero_score, few_score, lora_score)
        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

    print("\n" + "=" * 100)
    print("THREE-METHOD COMPARISON - WEIGHTED F1")
    print("=" * 100)
    print(f"{'Task':<30} {'Zero-shot':<12} {'Few-shot':<12} {'LoRA':<12} {'Best':<12}")
    print("-" * 100)

    # Label-based tasks - Weighted F1
    for task in ['Company', 'Sentiment', 'Industry', 'Coarse-Grained Event', 'Fine-Grained Event']:
        zero_score = zero_by_label[task]['f1_weighted']
        few_score = few_by_label[task]['f1_weighted']
        lora_score = lora_by_label[task]['f1_weighted']

        best = get_best_method(zero_score, few_score, lora_score)
        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

    print("-" * 100)

    # Variant-based tasks - Weighted F1
    for task in ['EFSA', 'Coarse-grained EFSA', 'Fine-grained EFSA', 'Entity-Level FSA']:
        zero_score = zero_by_variant[task]['f1_weighted']
        few_score = few_by_variant[task]['f1_weighted']
        lora_score = lora_by_variant[task]['f1_weighted']

        best = get_best_method(zero_score, few_score, lora_score)
        print(f"{task:<30} {zero_score:<12.4f} {few_score:<12.4f} {lora_score:<12.4f} {best:<12}")

compare_three_methods(df[:50], zero_df, few_df, lora_df)

THREE-METHOD COMPARISON - ACCURACY
Task                           Zero-shot    Few-shot     LoRA         Best        
----------------------------------------------------------------------------------------------------
Company                        0.7000       0.5000       0.9200       LoRA        
Sentiment                      0.7200       0.7400       0.8000       LoRA        
Industry                       0.4000       0.2600       0.3600       Zero-shot   
Coarse-Grained Event           0.8200       0.6200       0.7600       Zero-shot   
Fine-Grained Event             0.4600       0.3800       0.5200       LoRA        
----------------------------------------------------------------------------------------------------
EFSA                           0.1000       0.0800       0.0800       Zero-shot   
Coarse-grained EFSA            0.3200       0.1800       0.2800       Zero-shot   
Fine-grained EFSA              0.1000       0.0800       0.0800       Zero-shot   
Entity-Level FSA